# Maas Psilocybin Between-Subject FC and Brain-State Analysis

This notebook is a guided companion to `scripts/maas_psilbetween_fc_brain_states.py`. It intentionally analyzes only the between-subject psilocybin dataset, because the available psilocybin-within and THC-within time-series files contain duplicated condition arrays and are not valid for placebo-vs-drug contrasts.

The workflow produces publication-oriented functional connectivity and phase-coherence brain-state figures, plus CSV tables for statistical reporting. It also writes `ANALYSIS_GUIDE.md`, a plain-language methods and figure guide for collaborators.

## Analysis Design

- Dataset: `data/drugs_data/maas_psilbetween_ts_aal116_noGSR.mat`
- Groups: placebo and psilocybin, between-subject design
- Atlas: AAL116, no GSR time series
- FC: Pearson correlation per scan, Fisher-z edge statistics, Welch tests, Benjamini-Hochberg FDR
- Brain states: pooled phase-coherence patterns using the TVBToolkit `brain_act_legacy` BOLD pipeline, `k=5` shared states, occupancy and no-self transition summaries per scan
- Figure formats: PDF, SVG, and PNG
- Aesthetic: Wes Anderson-inspired Zissou palette, with deep sea blue for placebo and ochre for psilocybin
- Comparison colorbars are directional: more placebo on the negative side, more psilocybin on the positive side
- Figures with statistical tests mark FDR-significant effects where applicable
- Luppi-style FC topology outputs are included as an immediate adaptation; true PhiID STS/RTR analyses require a separate MATLAB PhiID run

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUT_DIR = PROJECT_ROOT / 'results' / 'maas_psilbetween_fc_brain_states'
FIG_DIR = OUT_DIR / 'figures'
TABLE_DIR = OUT_DIR / 'tables'

print(PROJECT_ROOT)
print(OUT_DIR)

## Run the Reproducible Workflow

This calls the single source-of-truth script. Re-run this cell whenever you change parameters or regenerate the figures.

In [ ]:
cmd = [sys.executable, str(PROJECT_ROOT / 'scripts' / 'maas_psilbetween_fc_brain_states.py')]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

## Key Statistical Tables

In [ ]:
fc_stats = pd.read_csv(TABLE_DIR / 'fc_subject_summary_stats.csv')
state_stats = pd.read_csv(TABLE_DIR / 'brain_state_occupancy_stats.csv')
edge_stats = pd.read_csv(TABLE_DIR / 'fc_edgewise_welch_fdr.csv')
macro_stats = pd.read_csv(TABLE_DIR / 'fc_macro_system_edge_summary.csv')

display(fc_stats)
display(state_stats[['state', 'placebo_mean', 'psilocybin_mean', 'delta_mean', 'p', 'q']])
display(edge_stats.head(10))
display(macro_stats.sort_values('delta_z_mean', ascending=False).head(10))

## Functional Connectivity Figures

In [ ]:
for name in [
    'fig01_qc_metadata.png',
    'fig02_fc_group_heatmaps.png',
    'fig03_fc_subject_summary.png',
    'fig04_fc_edgewise_volcano.png',
    'fig05_fc_macro_system_delta.png',
    'fig10_luppi_style_fc_topology.png',
    'fig11_luppi_style_fc_rank_gradients.png',
]:
    print(name)
    display(Image(filename=str(FIG_DIR / name)))

## Brain-State Figures

States are sorted by increasing mean phase-coherence centroid strength. The occupancy panel uses FDR-corrected q-values in the title annotation.

In [ ]:
for name in [
    'fig06_brain_state_centroids.png',
    'fig07_brain_state_occupancy.png',
    'fig08_brain_state_transitions.png',
    'fig09_brain_state_dwell_synchrony.png',
]:
    print(name)
    display(Image(filename=str(FIG_DIR / name)))

## Reporting Notes

- `scan_level_fc_brain_state_metrics.csv` contains one row per scan with metadata, FC summaries, dwell metrics, and global synchrony.
- `fc_edgewise_welch_fdr.csv` contains all 6,670 AAL116 edges with delta Fisher-z, Welch t-test p-values, and FDR q-values.
- `brain_state_occupancy_long.csv` and `brain_state_transition_long.csv` are the long-form brain-state outputs for custom statistics or plotting.
- `ANALYSIS_GUIDE.md` explains every analysis and figure in plain language.
- `logs/run_manifest.json` records parameter settings and source files for reproducibility.